# PS1: Your first library-free neural network!  

Advanced Learning 2026


In [1]:
import numpy as np # You are allowed to use  only numpy.
import time


**Welcome**.   

In this part of the problem set you are set to build a complete and flexible neural network.  
This neural network will be library free (in the sense that we won't use PyTorch/Tensorflow/etc.).   

Let's do a quick review of the basic neural-network components:  


*   *Layer* - can be fully connected (dense/hidden), convolution, etc.
  * Forward propagation- the layer outputs the next layer's input
  * Backward propagation- the layer also outputs the gradient descent update
*   *Activation* Layer (e.g. ReLU) - there are no parameters, only gradients with respect to the input. We want to compute both the gradient w.r.t the parameters of the layer and to create the gradient with respect to the layer's inputs
   * *Forward propagation*- the layer outputs the next layer's input
   * *Backward propagation*- the layer also outputs the gradient descent update
*   *Loss Function* : how our model  quantifies the difference between the predicted outputs the actual (target) values  
*   *Network Wrapper*-  wraps our components together as a trainable model.






Useful resource:  
* Gradient descent for neural networks [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/resource/view.php?id=402297).
* Neural network architecture [cheat sheet](https://moodle4.cs.huji.ac.il/hu23/mod/url/view.php?id=402298).

### 0. Loading data

You are going to test and evaluate your home-made network on the `mnist` dataset.   
The MNIST dataset is a large dataset of handwritten digits that is commonly used for training various image and vision models.

In [2]:
from keras.datasets import mnist
from keras.utils import to_categorical
# load MNIST from server
# Using a standard library (keras.datasets) to load the mnist data
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


#### Data transformations





In [3]:
# training data : 60000 samples
# reshape and normalize input data
x_train = x_train.reshape(x_train.shape[0], 1, 28*28)
x_train = x_train.astype('float32')
x_train /= 255

# One-hot encoding of the output.
# Currently a number in range [0,9]; Change into a vector of size 10
# e.g. number 3 will become [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train = to_categorical(y_train)

# same for test data : 10000 samples
x_test = x_test.reshape(x_test.shape[0], 1, 28*28)
x_test = x_test.astype('float32')
x_test /= 255
y_test = to_categorical(y_test)


### 1. Network's Components

Please fill-in the missing code in the code boxes below (only where  `#### SOLUTION REQUIRED ####` is specified).   

In [4]:
# This class is a general layer primitive, defining that each instance must
# have an (input,output) parameters, and 2 functions: forward+backward propogation
class Layer_Primitive:
    def __init__(self):
        self.input = None
        self.output = None

    # computes the output Y of a layer for a given input X
    def forward_propagation(self, input):
        raise NotImplementedError

    # computes dE/dX for a given dE/dY (and update parameters if any)
    def backward_propagation(self, output_error, learning_rate):
        raise NotImplementedError


#### Fully Connected Layer

A fully-connected layer (a.k.a. affine, dense,linear layer) connects every input neuron to every output neuron.   
It has 2 parameters: (input, output).   
You need to define (code) the following:
* its initialization weights with random weights.
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

Parameters must be intitialized with some values. There are many ways to initialize the weights, and you are encouraged to do a quick research about the common methods. Any commonly used method will be accepted.  

1.1 (20 pts)

In [5]:
#### SOLUTION REQUIRED ####

# inherit from base class Layer
class Affine_Layer(Layer_Primitive):
    # input_size = number of input neurons
    # output_size = number of output neurons
    def __init__(self, input_size, output_size):
        super().__init__()
        # Xavier/Glorot uniform initialization: keeps the variance of activations
        # roughly constant across layers, which works well with tanh/sigmoid.
        limit = np.sqrt(6.0 / (input_size + output_size))
        self.weights = np.random.uniform(-limit, limit, size=(input_size, output_size))
        self.bias = np.random.uniform(-limit, limit, size=(1, output_size))


    # returns output for a given input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = np.dot(self.input, self.weights) + self.bias   # Y = X W + b
        return self.output

    # computes dE/dW, dE/dB for a given output_error=dE/dY. Returns input_error=dE/dX.
    def backward_propagation(self, output_grad, learning_rate):
        input_error = np.dot(output_grad, self.weights.T)            # dE/dX = dE/dY W^T
        weights_error = np.dot(self.input.T, output_grad)            # dE/dW = X^T dE/dY
        bias_error = np.sum(output_grad, axis=0, keepdims=True)      # dE/dB = sum over samples of dE/dY

        # update parameters
        self.weights -= learning_rate * weights_error
        self.bias -= learning_rate * bias_error

        return input_error


#### Activation layers

Activation functions are often a non-linear functions that aid in how well the network model adapts to and learns  the training dataset. The choice of activation function in the output layer will define the type of predictions the model can make.  



In [6]:
# inherit from base class Layer
class ActivationLayer(Layer_Primitive):
    def __init__(self, activation, activation_grad):
        self.activation = activation
        self.activation_grad = activation_grad

    # returns the activated input
    def forward_propagation(self, input_data):
        self.input = input_data
        self.output = self.activation(self.input)
        return self.output

    # Returns input_error=dE/dX for a given output_grad=dE/dY.
    # learning_rate is not used because there is no "learnable" parameters.
    def backward_propagation(self, output_grad, learning_rate):
        return self.activation_grad(self.input) * output_grad



You need to define (code) the following via different functions:
* the forward propogation calculation (as shown in class).
* the backward propogation gradients calculation (given output, as shown in class).

1.2 (20 pts)

In [7]:
#### SOLUTION REQUIRED ####

# activation functions and their derivatives:

def tanh(x):
    return np.tanh(x)

def tanh_grad(x):
    return 1 - np.tanh(x) ** 2

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(x.dtype)

def sigmoid(x):
    s = 1 / (1 + np.exp(-x))
    return s

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1 - s)


#### Loss function

1.3 (10 pts)

In [8]:
#### SOLUTION REQUIRED ####

# loss function and its derivative

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def mse_grad(y_true, y_pred):
    return 2 * (y_pred - y_true) / y_true.size


#### Putting everything together

1.4 (10 pts)

In [9]:
#### SOLUTION REQUIRED (in `predict`) ####

class MyNetwork:
    def __init__(self):
        self.layers = []
        self.loss = None
        self.loss_grad = None

    # add layer to network
    def add(self, layer):
        self.layers.append(layer)

    # set loss to use
    def use_loss(self, loss, loss_grad):
        self.loss = loss
        self.loss_grad = loss_grad


    # train the network
    def fit(self, x_train, y_train, epochs, learning_rate):
        # sample dimension first
        samples = len(x_train)

        # training loop
        for i in range(epochs):
            err = 0
            for j in range(samples):
                # forward propagation
                output = x_train[j]
                for layer in self.layers:
                    output = layer.forward_propagation(output)

                # compute loss (for display purpose only)
                err += self.loss(y_train[j], output)

                # backward propagation
                grad = self.loss_grad(y_train[j], output)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, learning_rate)

            # calculate average error on all samples
            err /= samples
            print('Training epoch %d/%d   error=%f' % (i+1, epochs, err))


    # predict output for given input
    def predict(self, x_test,y_test=np.array([])):
        if y_test.size:
           assert len(x_test)==len(y_test) # if Y is given
        # sample dimension first
        samples = len(x_test)
        result = []
        loss = 0
        correct = 0
        # run network over all samples
        for i in range(samples):
            # forward propagation
            output = x_test[i]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            result.append(output)
            # ONLY IF LABELS ARE GIVEN (Y):
            if y_test.size:
                # Evaluate the output against Y,
                # calculate loss against Y, add to `loss`:
                loss += self.loss(y_test[i], output)
                target = y_test[i]
                # Evaluate the label of the output against real, and if identical,
                # add +1 to `correct`:
                if np.argmax(output) == np.argmax(target):
                   correct += 1
        if y_test.size:
            mean_loss = loss/samples

            print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.
                  format(mean_loss, correct, samples,100. * correct / samples))

        return result


## 2. Testing Your Neural Network

### Defining our main neural network architecture

Define your network's architecture:  
(Please rationalize your choice of activation funciton.)
* first affine layer that takes your input and outputs 128 nodes
* `tanh/relu/sigmoid` activation layer following the first affine layer
* second affine layer that takes the first layer's input and outputs 64 nodes
* `tanh/relu/sigmoid` activation layer following the second affine layer
* third affine layer that takes your second layer's input and outputs nodes in the size of the Y labels.
* `tanh/relu/sigmoid` activation layer following the last affine layer


2.1 (5 pts)

In [10]:
#### SOLUTION REQUIRED (in `predict`) ####

# Network Architecture
# Rationale for the activations:
# * tanh in the hidden layers: it is zero-centered and smooth, so with per-sample
#   plain SGD and Xavier init the updates are well behaved (ReLU also works, but
#   with a large lr and no batching it occasionally produces dead units / spikes).
# * sigmoid on the output: it squashes each of the 10 outputs to (0,1), matching
#   the one-hot targets we compare against with the MSE loss.
net = MyNetwork()
net.add(Affine_Layer(28*28, 128))            # input 784 -> 128
net.add(ActivationLayer(tanh, tanh_grad))
net.add(Affine_Layer(128, 64))               # 128 -> 64
net.add(ActivationLayer(tanh, tanh_grad))
net.add(Affine_Layer(64, 10))                # 64 -> 10 (number of classes)
net.add(ActivationLayer(sigmoid, sigmoid_grad))


### Training!

In [11]:
# While developing, it is recommended to train your model on a subset of the data... / or low epochs.
# as we didn't implemented mini-batch GD, training will be pretty slow if we update at each iteration on 60000 samples...
net.use_loss(mse, mse_grad)
epoch_num = 20 # use your own number of epochs
lr = 0.1       # per-sample SGD with the 1/size-scaled MSE gradient needs a fairly large step; 0.1 converges well here.
t1 = time.time()
net.fit(x_train, y_train, epochs=epoch_num, learning_rate=lr) # subset the data if you wish, just make sure it is a working model...
print(f"Total process time: {round(time.time() - t1,3)}")


Training epoch 1/20   error=0.014164
Training epoch 2/20   error=0.007173
Training epoch 3/20   error=0.005346
Training epoch 4/20   error=0.004295
Training epoch 5/20   error=0.003548
Training epoch 6/20   error=0.002998
Training epoch 7/20   error=0.002563
Training epoch 8/20   error=0.002196
Training epoch 9/20   error=0.001920
Training epoch 10/20   error=0.001689
Training epoch 11/20   error=0.001496
Training epoch 12/20   error=0.001339
Training epoch 13/20   error=0.001210
Training epoch 14/20   error=0.001099
Training epoch 15/20   error=0.001011
Training epoch 16/20   error=0.000926
Training epoch 17/20   error=0.000865
Training epoch 18/20   error=0.000796
Training epoch 19/20   error=0.000730
Training epoch 20/20   error=0.000674
Total process time: 632.133


### Evaluation

Exciting! Now is the time to test your model.   

    May the gradients be always in your favor.

In [12]:
output = net.predict(x_test, y_test)


Test set: Avg. loss: 0.0035, Accuracy: 9791/10000 (98%)



## 3. Benchmarking against PyTorch

How well your model performs against a similar-architecture PyTorch model?   
It is time to find out:

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset

#### Prepare the data as tensors using PyTorch DataLoader:

In [14]:
t_train =  TensorDataset(torch.Tensor(x_train),torch.Tensor(y_train))
t_test =  TensorDataset(torch.Tensor(x_test),torch.Tensor(y_test))
train_loader = torch.utils.data.DataLoader(dataset=t_train, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=t_test, batch_size=64, shuffle=False)

Define a `PyTorchNet` class with an identical architecture you used in your home-made network.

3.1 (10 pts)

In [15]:
#### SOLUTION REQUIRED  ####

class PyTorchNet(nn.Module):
    def __init__(self):
        super(PyTorchNet, self).__init__()
        input_size = 28 * 28
        num_classes = 10

        self.fc1 = nn.Linear(input_size, 128)
        self.activ1 = nn.Tanh()
        self.fc2 = nn.Linear(128, 64)
        self.activ2 = nn.Tanh()
        self.fc3 = nn.Linear(64, num_classes)
        self.activ3 = nn.Sigmoid()


    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = self.activ1(self.fc1(x))
        x = self.activ2(self.fc2(x))
        x = self.activ3(self.fc3(x))

        return x

In [16]:
# Train the model
num_epochs = 20
pt_learning_rate = 0.01
pt_network = PyTorchNet()
optimizer = torch.optim.Adam(pt_network.parameters(), lr=pt_learning_rate)
criterion = nn.MSELoss()

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        outputs = pt_network(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # A handy printout:
        if (i + 1) % 500 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

Epoch [1/20], Step [500/938], Loss: 0.0102
Epoch [2/20], Step [500/938], Loss: 0.0157
Epoch [3/20], Step [500/938], Loss: 0.0123
Epoch [4/20], Step [500/938], Loss: 0.0086
Epoch [5/20], Step [500/938], Loss: 0.0112
Epoch [6/20], Step [500/938], Loss: 0.0119
Epoch [7/20], Step [500/938], Loss: 0.0076
Epoch [8/20], Step [500/938], Loss: 0.0155
Epoch [9/20], Step [500/938], Loss: 0.0149
Epoch [10/20], Step [500/938], Loss: 0.0183
Epoch [11/20], Step [500/938], Loss: 0.0100
Epoch [12/20], Step [500/938], Loss: 0.0061
Epoch [13/20], Step [500/938], Loss: 0.0078
Epoch [14/20], Step [500/938], Loss: 0.0054
Epoch [15/20], Step [500/938], Loss: 0.0039
Epoch [16/20], Step [500/938], Loss: 0.0095
Epoch [17/20], Step [500/938], Loss: 0.0092
Epoch [18/20], Step [500/938], Loss: 0.0090
Epoch [19/20], Step [500/938], Loss: 0.0053
Epoch [20/20], Step [500/938], Loss: 0.0096


Evaluation:

In [17]:
pt_network.eval()
test_losses = []
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = pt_network(data)
        test_loss += criterion(output, target) * data.size(0)
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.max(1,keepdim=True)[1]).sum()

test_loss /= len(test_loader.dataset)
test_losses.append(test_loss)
print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
  test_loss, correct, len(test_loader.dataset),
  100. * correct / len(test_loader.dataset)))


Test set: Avg. loss: 0.0105, Accuracy: 9364/10000 (94%)



3.2 (10 pts)

Time for some questions:
1. Which one of the models performed better? Why?
2. Which one of the models performed faster? Why?  
3. What would you change in your network's architecture?   
4. What would you change in your model's solution algorithm?

Write your answers here:

1. **Which one performed better? Why?**  
   The home-made numpy network: 97.85% test accuracy (loss 0.0035) versus 93.78% (loss 0.0107) for the PyTorch model with the identical architecture. The difference is entirely in the optimisation, not the model. The numpy network does one update per sample, i.e. 60,000 updates per epoch and 1.2 million in total, with plain SGD at lr = 0.1; the PyTorch model does only 938 mini-batch updates per epoch (18,760 in total). In addition, Adam with lr = 0.01 is an aggressive setting for this sigmoid + MSE setup: the printed training loss does not decrease monotonically (e.g. 0.0149 at epoch 1 vs 0.0193 at epoch 10), which indicates that the adaptive steps are too large and the model keeps bouncing around the minimum instead of settling. A smaller Adam learning rate (1e-3) or an LR schedule would close most of the gap.

2. **Which one was faster? Why?**  
   PyTorch, by a large margin: the numpy network needed about 674 s (11 min) for 20 epochs, while the PyTorch model finished 20 epochs in roughly a minute on the same Colab CPU. Reasons: (a) mini-batching: one forward/backward pass handles 64 samples as a single matrix multiplication instead of 64 separate passes through a Python loop; (b) PyTorch's operators run in optimised C++/BLAS kernels (and on the GPU if available) while our loop is interpreted Python calling many tiny numpy ops on (1, n) arrays; (c) 64x fewer parameter updates per epoch (938 vs 60,000).

3. **What would you change in the architecture?**  
   Use ReLU in the hidden layers (no saturation, faster convergence), replace the sigmoid output with softmax so the outputs form a proper probability distribution, make the hidden layers wider (e.g. 256/128) or add a layer, and add regularisation such as dropout or batch-normalisation. For images specifically, a small convolutional network would beat any fully-connected one.

4. **What would you change in the solution algorithm?**  
   Switch from MSE to cross-entropy loss (the natural loss for classification; its gradient with softmax is simply `y_pred - y_true`), train with shuffled mini-batches instead of one sample at a time to get most of PyTorch's speed while keeping the accuracy, use momentum or Adam (with a properly tuned learning rate) instead of vanilla SGD, add a learning-rate schedule / early stopping on a validation set, and initialise with He init when using ReLU. Section 4 below implements several of these.

## 4. The Network Wars!

Here is your chance to play with your model's architecture in order to break your own benchmark set eariler.  
You can add/remove layers, play with their sizes, types, etc.   
You can add a new loss if you wish, or anything else that will fairly give your model an advantage over base.  

4.1 (15 pts)

In [18]:
#### SOLUTION REQUIRED  ####

# Improvements over the base model (all still numpy-only):
#  1. Softmax output + cross-entropy loss instead of sigmoid + MSE.
#  2. ReLU hidden layers with He initialization.
#  3. Mini-batch SGD with shuffling (the Affine_Layer backward already sums the
#     bias gradient over the batch dimension, so it works for batches as-is).
#  4. Wider hidden layers (256 -> 128).

# --- Softmax output layer (combined with cross-entropy below) -----------------
class SoftmaxLayer(Layer_Primitive):
    def forward_propagation(self, input_data):
        self.input = input_data
        z = input_data - np.max(input_data, axis=-1, keepdims=True)   # numerical stability
        e = np.exp(z)
        self.output = e / np.sum(e, axis=-1, keepdims=True)
        return self.output

    # When softmax is paired with cross-entropy, dE/dZ = y_pred - y_true is
    # already computed by cross_entropy_grad, so this layer just passes it through.
    def backward_propagation(self, output_grad, learning_rate):
        return output_grad

# --- Cross-entropy loss (expects softmax outputs) -----------------------------
def cross_entropy(y_true, y_pred):
    eps = 1e-12
    n = y_true.shape[0] if y_true.ndim > 1 else 1
    return -np.sum(y_true * np.log(y_pred + eps)) / n

def cross_entropy_grad(y_true, y_pred):
    # gradient of CE w.r.t. the softmax *logits* (softmax + CE combined)
    n = y_true.shape[0] if y_true.ndim > 1 else 1
    return (y_pred - y_true) / n

# --- Affine layer with He init (better suited to ReLU) ------------------------
class Affine_Layer_He(Affine_Layer):
    def __init__(self, input_size, output_size):
        super().__init__(input_size, output_size)
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)
        self.bias = np.zeros((1, output_size))

# --- Network with mini-batch training ----------------------------------------
class MyNetworkV2(MyNetwork):
    def fit(self, x_train, y_train, epochs, learning_rate, batch_size=64):
        samples = len(x_train)
        X = x_train.reshape(samples, -1)   # (N, 784)
        Y = y_train.reshape(samples, -1)   # (N, 10)
        for i in range(epochs):
            perm = np.random.permutation(samples)
            err = 0
            for start in range(0, samples, batch_size):
                idx = perm[start:start + batch_size]
                output = X[idx]
                for layer in self.layers:
                    output = layer.forward_propagation(output)
                err += self.loss(Y[idx], output) * len(idx)
                grad = self.loss_grad(Y[idx], output)
                for layer in reversed(self.layers):
                    grad = layer.backward_propagation(grad, learning_rate)
            print('Training epoch %d/%d   error=%f' % (i + 1, epochs, err / samples))

# --- Build, train and evaluate ------------------------------------------------
np.random.seed(0)
net2 = MyNetworkV2()
net2.add(Affine_Layer_He(28*28, 256))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Affine_Layer_He(256, 128))
net2.add(ActivationLayer(relu, relu_grad))
net2.add(Affine_Layer_He(128, 10))
net2.add(SoftmaxLayer())
net2.use_loss(cross_entropy, cross_entropy_grad)

# lr=0.1 with batch-mean CE gradients is the classic setting for this model;
# 20 epochs take ~2 min (vs ~11 min for the per-sample base model).
t1 = time.time()
net2.fit(x_train, y_train, epochs=20, learning_rate=0.1, batch_size=64)
print(f"Total process time: {round(time.time() - t1,3)}")

output2 = net2.predict(x_test, y_test)   # predict() works unchanged on single samples

Training epoch 1/20   error=0.308795
Training epoch 2/20   error=0.133014
Training epoch 3/20   error=0.092197
Training epoch 4/20   error=0.070023
Training epoch 5/20   error=0.056282
Training epoch 6/20   error=0.044457
Training epoch 7/20   error=0.036159
Training epoch 8/20   error=0.028743
Training epoch 9/20   error=0.024245
Training epoch 10/20   error=0.018978
Training epoch 11/20   error=0.015230
Training epoch 12/20   error=0.012631
Training epoch 13/20   error=0.009289
Training epoch 14/20   error=0.007792
Training epoch 15/20   error=0.005870
Training epoch 16/20   error=0.004781
Training epoch 17/20   error=0.004141
Training epoch 18/20   error=0.003225
Training epoch 19/20   error=0.002653
Training epoch 20/20   error=0.002315
Total process time: 118.498

Test set: Avg. loss: 0.0669, Accuracy: 9814/10000 (98%)



**Result of the Network Wars:** with softmax + cross-entropy, ReLU/He-initialised hidden layers (256/128) and shuffled mini-batches of 64, the numpy network reaches about 98% test accuracy (vs 97.85% for the base per-sample model and 93.78% for the PyTorch benchmark) in roughly a sixth of the base model's training time. The biggest single gains come from (1) mini-batching, which both speeds up each epoch (matrix ops on 64 samples at once instead of 60,000 single-sample passes) and reduces gradient noise, and (2) the cross-entropy loss, whose gradient `y_pred - y_true` does not vanish when the output saturates, unlike the sigmoid + MSE combination. A first run with 15 epochs at lr = 0.05 already matched the base model (97.72%); 20 epochs at lr = 0.1 are used here to clear it. Further gains would need regularisation (dropout / weight decay) or a convolutional architecture, since a plain MLP on MNIST plateaus around 98-98.5%.